# 03. Видео VAE: posterior, compression, causality и scaling

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

**Результат:** вывести VAE objective, обучить маленький codec и проверить causal Conv3d/latent normalization

**Перед началом:** Lecture 4 / Lab 3; подробная теория VAE в K5 02. [Порядок всех материалов](../../curriculum/CONTENTS.md).
[Курс K6](../README.md) · [Покомпонентная карта](../COMPONENT_GUIDE.md) · [Предыдущий опыт K5](../../kandinsky-5/notebooks/02_vae.ipynb).

Основной архитектурный источник: [Kandinsky 6 на Hugging Face Papers](https://huggingface.co/papers/2610.05608),
[полный отчёт, §3.1](https://arxiv.org/html/2610.05608v1). Открывайте статью рядом с notebook.
Мы различаем опубликованную архитектуру, конкретный checkpoint config и результат нашего опыта.

**Код для сравнения (закреплённая revision `7ceeefad`):**
- [`core/components/hunyuan_vae.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/hunyuan_vae.py)
- [`core/components/vae_video.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/components/vae_video.py)
- [`core/algo/postprocess_video.py`](https://github.com/kandinskylab/kandinsky-6/blob/7ceeefad7fb625011f7db602c43893a989f19c15/kandinsky/core/algo/postprocess_video.py)

Run All выполняет небольшие CPU-опыты. Случайная уменьшенная сеть проверяет вычисления;
pretrained генерацию, качество голоса и производительность M3 она не измеряет.
Большие веса не скачиваются, ComfyUI jobs не отправляются. В Colab первая ячейка получает
исходники курса из GitHub, если их ещё нет; пакеты не устанавливает. На Mac используется существующий checkout.
Ниже объяснение идёт перед каждым опытом.


In [ ]:
from pathlib import Path
import json, sys, os, math
for parent in (Path.cwd(), *Path.cwd().parents):
    candidates = (parent, parent / 'kandinsky-6')
    found = next((p for p in candidates if (p / 'labkit/components.py').is_file()), None)
    if found is not None:
        ROOT = found.resolve()
        break
else:
    if not Path('/content').is_dir():
        raise FileNotFoundError('Откройте notebook внутри checkout generative-ai-research.')
    import subprocess
    checkout=Path('/content/generative-ai-research-k6-components')
    if not checkout.exists():
        subprocess.run(['git','clone','--depth','1','--branch',
            os.environ.get('K6_LABS_REF','kandinsky-6-components'),
            'https://github.com/vladislav-vasilenko/generative-ai-research.git',str(checkout)],check=True)
    ROOT=checkout/'kandinsky-6'
    if not (ROOT/'labkit/components.py').is_file():
        raise FileNotFoundError('В выбранной revision нет component labs; проверьте K6_LABS_REF.')
sys.path.insert(0, str(ROOT))
os.environ.setdefault('MPLCONFIGDIR', str(ROOT / 'artifacts/component_labs/mpl-cache'))
import numpy as np
import matplotlib.pyplot as plt
import torch
from torch import nn
import torch.nn.functional as F
from labkit.components import (config, native, show_source, native_causal_conv,
    tiny_model, tiny_inputs, activate_gates, fresh_forward, write_metrics, figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
np.random.seed(42)
LAB = '03_video_vae'
metrics = {}
print('CPU teaching mode; PyTorch', torch.__version__)


## 1. Зачем распределение вместо одной точки

Encoder оценивает Gaussian posterior с диагональной covariance. Декодер задаёт likelihood наблюдения.
Reparameterization отделяет случайность от обучаемых mean/std:

$$
q_\phi(z\mid x)=\mathcal N(\mu_\phi(x),\mathrm{diag}(\exp(\ell_\phi(x)))),\qquad
z=\mu_\phi(x)+\exp(\ell_\phi(x)/2)\epsilon.
$$

Negative ELBO состоит из reconstruction NLL и KL к prior. Для standard Gaussian prior:

$$
\mathrm{KL}(q\Vert p)=\frac12\sum_j\left(\mu_j^2+\exp(\ell_j)-1-\ell_j\right).
$$

MSE соответствует Gaussian reconstruction likelihood при фиксированной дисперсии с соответствующим
коэффициентом и константой. Меняя вес KL или редукцию MSE, вы меняете objective. Ниже честно используем
**учебный beta-VAE objective** `mean_MSE + 0.001 * mean_batch(sum_latent_KL)`,
а не утверждаем, что воспроизвели все perceptual/adversarial losses промышленного Hunyuan VAE.


In [ ]:
vae_cfg=config('vae')
T,H,W=121,480,864
latent_shape=((T-1)//4+1,H//8,W//8,vae_cfg['latent_channels'])
assert latent_shape==(31,60,108,16)
print('RGB video -> latent THWC:',(T,H,W,3),'->',latent_shape)
metrics['full_video_latent_shape']=list(latent_shape)
print('Scaling factor:',vae_cfg['scaling_factor'])


## 2. Миниатюрное обучение с известными данными

Создадим 12 коротких клипов движущегося Gaussian blob. Encoder/decoder — маленькие MLP,
чтобы видеть posterior и loss на CPU. Они не заменяют causal convolutional Hunyuan architecture.
Разделение цели и реализации позволяет сначала проверить математику.


In [ ]:
y,x=torch.meshgrid(torch.linspace(-1,1,8),torch.linspace(-1,1,8),indexing='ij')
clips=torch.stack([torch.stack([torch.exp(-((x-(t/4-0.5)*0.8)**2+(y-s)**2)/0.12)
                               for t in range(5)]) for s in torch.linspace(-.5,.5,12)])
flat=clips.flatten(1)
class MiniVAE(nn.Module):
    def __init__(self):
        super().__init__();self.encoder=nn.Sequential(nn.Linear(320,32),nn.SiLU(),nn.Linear(32,16))
        self.decoder=nn.Sequential(nn.Linear(8,32),nn.SiLU(),nn.Linear(32,320),nn.Sigmoid())
    def forward(self,x):
        mu,logvar=self.encoder(x).chunk(2,-1)
        z=mu+torch.exp(logvar/2)*torch.randn_like(mu)
        return self.decoder(z),mu,logvar
vae=MiniVAE();opt=torch.optim.Adam(vae.parameters(),lr=.01);history=[]
for step in range(100):
    recon,mu,logvar=vae(flat)
    reconstruction=F.mse_loss(recon,flat)
    kl=.5*(mu.square()+logvar.exp()-1-logvar).sum(-1).mean()
    loss=reconstruction+.001*kl
    opt.zero_grad();loss.backward();opt.step()
    history.append([loss.item(),reconstruction.item(),kl.item()])
assert np.isfinite(history).all() and history[-1][0]<history[0][0]
metrics['toy_objective_initial']=history[0][0];metrics['toy_objective_final']=history[-1][0]
with torch.no_grad():
    mu,logvar=vae.encoder(flat).chunk(2,-1);mean_recon=vae.decoder(mu).reshape_as(clips)
fig,axes=plt.subplots(1,3,figsize=(10,3));axes[0].plot(np.array(history)[:,:2])
axes[0].legend(['objective','MSE']);axes[1].imshow(clips[0,2]);axes[2].imshow(mean_recon[0,2])
axes[1].set_title('Input');axes[2].set_title('Decode posterior mean')
fig.tight_layout();fig.savefig(figure_path(LAB,'vae_training'));plt.show()


## 3. Causal Conv3d и нормализация латентов

Кодек с causal temporal padding не использует будущие кадры. Возьмём **оригинальное определение**
`HunyuanVideoCausalConv3d`, изолированно от тяжёлых Diffusers imports, и изменим только будущее.
Проверяем конкретный слой при kernel=3/stride=1; это не доказательство causality всего генератора:
video attention в DiT может видеть всю последовательность.

DiT обучен на масштабированных латентах. При encode умножаем на scale, перед decode делим.
Пропущенный scale меняет distribution, хотя shape остаётся правильной.


In [ ]:
Causal=native_causal_conv();conv=Causal(1,1,kernel_size=3,bias=False)
video=torch.randn(1,1,6,5,5);future=video.clone();future[:,:,4:]+=100
with torch.no_grad():
    left=conv(video);right=conv(future)
causal_error=(left[:,:,:4]-right[:,:,:4]).abs().max().item();assert causal_error==0
z=torch.randn(1,16,3,4,4);scale=vae_cfg['scaling_factor']
roundtrip=((z*scale)/scale-z).abs().max().item()
metrics['native_causal_conv_past_error']=causal_error;metrics['latent_scale_roundtrip_error']=roundtrip
print('Causal past error:',causal_error,'scale roundtrip:',roundtrip)


## Самостоятельное исследование

1. Сравните beta=0, 0.001, 0.1: MSE, KL, posterior variance. Не делайте вывод по одному loss.
2. Сопоставьте 33/121 RGB frames с latent frames. Почему формула содержит T-1, а не просто T/4?
3. Измените последний кадр и сравните past outputs causal и обычной symmetric convolution.
4. Для настоящего VAE reconstruction используйте ComfyUI C4 и существующий Hunyuan codec после resource gate.

Перед изменением запишите гипотезу. Сохраните baseline, меняйте один фактор и повторяйте по 3–5 seeds. Файл metrics.json содержит измерение конкретного запуска, а не статистическую оценку качества модели.


In [ ]:
write_metrics(LAB, metrics)


## Критерий понимания

Вы различаете posterior mean/sample, reconstruction loss/KL, pixel/latent space и причинность кодека/DiT.
Другой VAE с теми же размерами не гарантирует совместимость с pretrained CrossDiT.

[Вернуться к маршруту K6](../README.md) · [Полное содержание](../../curriculum/CONTENTS.md) · [Шаблон отчёта](../../curriculum/EXPERIMENT_TEMPLATE.md).


[← 02. Текст: tokenizer, Qwen, CLIP и два refiner](02_text_conditioning.ipynb) · [04. Аудио: waveform → mel → VAE → vocoder →](04_audio_vae_and_vocoder.ipynb)
